# RecoAI · E5 임베딩 결합 기반 업종 분류 실험

## 실험 개요

- 분류 대상: 54개 업종
- 원본 출력 기준 입력 데이터: 2,601개 질문
- 질문 임베딩과 업종명 임베딩: 각각 1,024차원
- 분류기: 결합 벡터 2,048차원 → Linear(512) → ReLU → Linear(1), 각 업종의 점수 계산
- 학습 대상: 사전 계산한 E5 임베딩을 입력받는 FC 분류기. E5 자체를 미세조정하는 학습은 아닙니다.

## 평가 해석 — 반드시 확인

별도의 validation/test 분리 없이 전체 입력 데이터를 학습에 사용했습니다. 마지막 epoch의 `acc: 0.9946`, `f1: 0.9961`은 **학습 중 배치 예측을 모아 계산한 지표**입니다. 고정된 최종 모델을 독립 데이터로 평가한 수치가 아니며, 일반화 성능이나 기존 방식 대비 개선율로 사용할 수 없습니다.

뒤쪽 `test_questions`는 수동으로 작성한 동작 확인 질문입니다. 독립 테스트셋이나 정량 평가로 해석하지 않습니다. 학습 시드도 고정되지 않아 재실행 결과가 달라질 수 있습니다.

필요 입력 파일:
- `questions_reward_general_1216.csv`
- `questions_reward_store_1216.csv`
- `additional_question_1223.txt`

저장 결과: `category_classifier.pth`, `category_embeddings.npy`.

## 설계 배경과 한계

당시 메모에는 GPU 메모리와 응답 지연을 고려해 기존 임베딩 모델 활용을 검토했다고 기록되어 있습니다. 단순 코사인 유사도 대신 질문·업종명 벡터의 결합을 학습하는 FC 분류기를 구현했습니다. 실제 메모리 절감률이나 지연 개선은 측정하지 않았습니다.

유사 업종 구분과 질문 데이터 부족·불균형을 검토했지만, 이를 독립 평가로 검증한 것은 아닙니다. 현재 구현은 단일 정답 업종을 선택하는 multiclass 분류입니다. Top-2 실험도 multilabel 학습을 의미하지 않습니다. softmax 확률의 전체 평균은 항상 약 `1/54`이므로 신뢰도 지표가 아닙니다.


## 공개용 정리 안내

이 문서는 2024년 RecoAI 프로젝트에서 보존된 실험 노트북을 2026-09-13에 공개용으로 정리한 자료입니다. 최종 배포 코드와 동일한 버전인지는 확정하지 않았습니다.

- E5(`intfloat/multilingual-e5-large-instruct`) 임베딩과 FC 분류기를 사용합니다.
- 당시 학습·추론 알고리즘, 하이퍼파라미터 및 파일 경로는 보존했습니다. 이번 정리에서는 재학습하거나 성능을 새로 측정하지 않았습니다.
- 아래 출력은 원본에 저장된 실행 기록입니다. 다운로드 진행 표시, 반복 경고 및 중복 대량 출력만 제거했습니다.
- Colab 원본 식별자·작성 태그·위젯 메타데이터를 제거하고, 5개 업종으로 잘못 적힌 주석을 54개로 정정했습니다.
- 파일명에 포함된 `two_tower`는 당시 실험명입니다. 실제 구현은 동일한 E5 모델로 질문과 업종명을 인코딩한 후 벡터를 결합하여 업종별 점수를 계산하는 구조입니다.

## 실행 범위

당시 Google Colab 및 Google Drive 환경의 경로가 남아 있습니다. 이 노트북만으로 즉시 전체 실행되지는 않습니다. 해당 입력 자료와 가중치를 준비한 뒤 경로를 조정해야 합니다.

필요 라이브러리: PyTorch, sentence-transformers, NumPy, Pandas, scikit-learn. 가맹점 추출에는 RapidFuzz가 추가로 필요합니다. 정확한 전체 실행 환경 버전은 고정되어 있지 않습니다.

가중치 로딩은 원본 `torch.load` 코드를 유지했습니다. 신뢰할 수 있는 본인의 가중치만 사용하고, CPU 환경에서 재실행할 경우 `map_location` 설정을 검토해야 합니다.


In [ ]:
# prompt: 구글드라이브연결

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sentence_transformers import SentenceTransformer, models, losses
from torch.utils.data import Dataset, DataLoader
import numpy as np
import math
import random
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


#### 1. 데이터셋 구성

In [ ]:
# 54개 카테고리
CATEGORIES = ['LPG충전소', 'OTT', '간편결제', '게임', '공과금', '공연', '기업형슈퍼마켓', '대중교통',
       '대형마트', '드럭스토어', '디지털구독', '렌탈', '리조트', '멤버십구독', '면세점', '미용', '미용실',
       '반려동물', '배달', '백화점', '보험', '세탁소', '쇼핑', '스포츠', '아울렛', '여행', '영화관',
       '온라인서점', '온라인쇼핑', '우체국', '웹툰', '음식점', '의료', '전기차충전소', '제과아이스크림',
       '주유소', '주차장', '차량정비소', '창고형할인매장', '카페', '콘도', '타이어샵', '택시', '테마파크',
       '통신사', '패션', '편의점', '학습지', '학원', '항공사', '해외대중교통', '호텔', '호텔음식점',
       '홈쇼핑']

In [ ]:
# CSV 파일 경로
file_path = '/content/drive/MyDrive/Project_final/data/questions_reward_general_1216.csv'

# CSV 파일을 DataFrame으로 읽기 (첫 번째 행을 컬럼명으로 사용)
df_general = pd.read_csv(file_path)

# DataFrame 확인
print(df_general.head())

  category                            question          tool
0     간편결제      카카오페이로 결제할 때 할인 혜택이 많은 카드 추천해줘  Reward_logic
1     간편결제  네이버페이로 자주 결제하는데 혜택 받을 수 있는 카드 추천해줘  Reward_logic
2     간편결제         삼성페이 사용 시 할인 혜택이 좋은 카드가 뭐야?  Reward_logic
3     간편결제           페이코로 결제할 때 혜택이 많은 카드 추천해줘  Reward_logic
4     간편결제    SSG페이로 쇼핑할 때 할인 받을 수 있는 카드가 있을까?  Reward_logic


In [ ]:
# CSV 파일 경로
file_path = '/content/drive/MyDrive/Project_final/data/questions_reward_store_1216.csv'

# CSV 파일을 DataFrame으로 읽기 (첫 번째 행을 컬럼명으로 사용)
df_store = pd.read_csv(file_path)

# DataFrame 확인
print(df_store.head())

  category   store                         question          tool
0      공과금    전기요금    전기요금 자동이체 시 할인 혜택이 있는 카드 추천해줘  Reward_logic
1      공과금    전기요금  전기요금 납부할 때 포인트 적립이 잘 되는 카드가 뭐야?  Reward_logic
2      공과금    전기요금   전기요금 고지서를 줄일 수 있는 할인 혜택 카드 알려줘  Reward_logic
3      공과금  도시가스요금   도시가스 요금 납부 시 할인 혜택이 좋은 카드 추천해줘  Reward_logic
4      공과금  도시가스요금       가스비 절약할 수 있는 카드 혜택 있으면 알려줘  Reward_logic


In [ ]:
# CSV 파일 경로
file_path = '/content/drive/MyDrive/Project_final/data/additional_question_1223.txt'

# CSV 파일을 DataFrame으로 읽기 (첫 번째 행을 컬럼명으로 사용)
df_add = pd.read_csv(file_path, delimiter='\t')

# DataFrame 확인
print(df_add.head())

  category                                           question
0     반려동물  우리 집 강아지가 편식을 해서 사료를 자주 바꾸는데, 구매 시 할인 혜택 좋은 카드...
1     반려동물  동물병원에서 수술비 부담이 큰 편인데, 결제 시 포인트가 많이 쌓이는 카드 추천받고...
2     반려동물         강아지 슬개골 탈구 치료에 들어가는 비용을 절약할 수 있는 카드가 궁금해요.
3     반려동물            고양이 중성화 수술 같은 대형 지출에 캐시백 혜택이 있는 카드 있나요?
4     반려동물             반려동물 장난감 구매할 때 매번 할인받을 수 있는 카드가 필요합니다.


In [ ]:
# 공통된 컬럼만 추출
df_general_subset = df_general[['category', 'question']]
df_store_subset = df_store[['category', 'question']]

# 세로로 이어붙이기
combined_df = pd.concat([df_general_subset, df_store_subset, df_add], ignore_index=True)

# 결과 출력
print(combined_df)

     category                               question
0        간편결제         카카오페이로 결제할 때 할인 혜택이 많은 카드 추천해줘
1        간편결제     네이버페이로 자주 결제하는데 혜택 받을 수 있는 카드 추천해줘
2        간편결제            삼성페이 사용 시 할인 혜택이 좋은 카드가 뭐야?
3        간편결제              페이코로 결제할 때 혜택이 많은 카드 추천해줘
4        간편결제       SSG페이로 쇼핑할 때 할인 받을 수 있는 카드가 있을까?
...       ...                                    ...
2596    디지털구독  디지털 음악 스트리밍 구독 결제에 적합한 카드를 추천받고 싶습니다.
2597    디지털구독          멜론을 자주 사용할 때 적합한 신용카드가 궁금합니다.
2598    디지털구독         지니뮤직 이용 시 적합한 신용카드를 추천받고 싶습니다.
2599    디지털구독         FLO 정기 결제 혜택이 많은 신용카드는 어떤 건가요?
2600    디지털구독      스포티파이 구독 서비스를 위한 최적의 신용카드는 무엇인가요?

[2601 rows x 2 columns]


In [ ]:
# category 컬럼의 값별 개수 확인
category_counts = combined_df['category'].value_counts()

# 개수의 평균값과 중위값 계산
counts_mean = category_counts.mean()
counts_median = category_counts.median()

# 결과 출력
print("Category별 개수:")
print(category_counts)
print("\n개수의 평균값:", counts_mean)
print("개수의 중위값:", counts_median)

Category별 개수:
category
음식점        325
OTT        127
대형마트       119
반려동물       110
창고형할인매장    107
호텔          96
쇼핑          95
디지털구독       95
카페          95
온라인쇼핑       88
멤버십구독       82
패션          71
간편결제        65
제과아이스크림     52
홈쇼핑         47
대중교통        46
통신사         41
백화점         39
주유소         38
스포츠         37
아울렛         35
의료          33
배달          32
편의점         31
드럭스토어       31
면세점         29
렌탈          29
온라인서점       29
웹툰          29
학습지         29
택시          29
공과금         28
항공사         28
게임          27
기업형슈퍼마켓     27
공연          26
여행          26
LPG충전소      26
영화관         23
해외대중교통      23
리조트         22
타이어샵        22
미용          22
우체국         21
보험          20
미용실         20
세탁소         20
전기차충전소      20
주차장         18
학원          18
차량정비소       16
콘도          14
호텔음식점       12
테마파크        11
Name: count, dtype: int64

개수의 평균값: 48.166666666666664
개수의 중위값: 29.0


In [ ]:
# 카테고리를 숫자로 매핑할 딕셔너리 생성
cat_to_idx = {cat: idx for idx, cat in enumerate(CATEGORIES)}

# (question, 숫자로 변환된 category) 형태로 리스트 변환
training_data = [
    (row["question"], cat_to_idx[row["category"]])
    for _, row in combined_df.iterrows()
]

print(training_data)
# 출력 예시:
# [
#   ('질문 예시1', 0),
#   ('질문 예시2', 53),
#   ('질문 예시3', 2)
# ]

In [ ]:
class QuestionCategoryDataset(Dataset):
    def __init__(self, data, model):
        """
        data: [(question_text, label_idx), ...]
        model: SentenceTransformer
        """
        self.data = data
        self.model = model

        # 질문 임베딩을 미리 구해둠
        self.samples = []
        for (question_text, label_idx) in self.data:
            q_emb = self.model.encode(question_text, convert_to_tensor=True)
            self.samples.append((q_emb, label_idx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        q_emb, label_idx = self.samples[idx]
        return q_emb, torch.tensor(label_idx, dtype=torch.long)

In [ ]:
# 첫 번째 타워(질문 임베딩), 두 번째 타워(카테고리 임베딩)는 동일한 모델
embedding_model = SentenceTransformer("intfloat/multilingual-e5-large-instruct").to(device)

# 카테고리 임베딩 계산
with torch.no_grad():
    category_embeddings = embedding_model.encode(CATEGORIES, convert_to_tensor=True)
    category_embeddings = category_embeddings.to(device)  # (54, 1024)

In [ ]:
train_dataset = QuestionCategoryDataset(data=training_data, model=embedding_model)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)

#### 1-1. 카테고리간 유사도 구하기

In [ ]:
online = embedding_model.encode('온라인쇼핑', convert_to_tensor=True)

In [ ]:
shopping = embedding_model.encode('쇼핑', convert_to_tensor=True)

In [ ]:
# prompt: oline과 shopping의 코사인 유사도를 구하는 코드

import torch

def cosine_similarity(a, b):
  """Calculates the cosine similarity between two tensors."""
  return torch.nn.functional.cosine_similarity(a, b, dim=0)

similarity = cosine_similarity(online, shopping)
print(f"The cosine similarity between '온라인쇼핑' and '쇼핑' is: {similarity}")

The cosine similarity between '온라인쇼핑' and '쇼핑' is: 0.9467802047729492


In [ ]:
online_space = embedding_model.encode('온라인 쇼핑', convert_to_tensor=True)

In [ ]:
similarity = cosine_similarity(online_space, shopping)
print(f"The cosine similarity between '온라인 쇼핑' and '쇼핑' is: {similarity}")

The cosine similarity between '온라인 쇼핑' and '쇼핑' is: 0.964637279510498


In [ ]:
CATEGORIES[46]

'편의점'

In [ ]:
high_similarity_list = []

for i in range(54):
  for j in range(i+1, 54):
    if i != j:
      similarity = cosine_similarity(category_embeddings[i], category_embeddings[j])
      if similarity > 0.9:
        high_similarity_list.append((CATEGORIES[i], CATEGORIES[j], similarity))

In [ ]:
for line in high_similarity_list:
  print(line)

### 2. train 코드

In [ ]:
class CategoryClassifier(nn.Module):
    def __init__(self, emb_dim=1024, hidden_dim=512):
        super().__init__()
        self.fc1 = nn.Linear(emb_dim * 2, hidden_dim)
        self.relu = nn.ReLU()
        # 카테고리별로 1개의 로짓만 예측 -> out_features=1
        self.fc2 = nn.Linear(hidden_dim, 1)

    def forward(self, q_emb_batch, cat_emb_all):
        """
        q_emb_batch: (batch, 1024)
        cat_emb_all: (54, 1024)

        return: (batch, 54)  # 54개 카테고리에 대한 로짓
        """
        batch_size = q_emb_batch.size(0)
        num_categories = cat_emb_all.size(0)  # 54
        # (batch, 1, 1024)
        q_emb_batch = q_emb_batch.unsqueeze(1)
        # (batch, 54, 1024)
        q_emb_batch = q_emb_batch.expand(batch_size, num_categories, q_emb_batch.size(-1))

        # (1, 54, 1024)
        cat_emb_all = cat_emb_all.unsqueeze(0)
        # (batch, 54, 1024)
        cat_emb_all = cat_emb_all.expand(batch_size, num_categories, cat_emb_all.size(-1))

        # Concatenate -> (batch, 54, 2048)
        x = torch.cat([q_emb_batch, cat_emb_all], dim=-1)

        # reshape -> (batch*54, 2048)
        x = x.view(batch_size * num_categories, -1)

        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)  # (batch*54, 1)

        # 다시 (batch, 54)로 만들기
        x = x.view(batch_size, num_categories)
        return x  # (batch, 54) => CrossEntropyLoss에 바로 적용

In [ ]:
# 모델 초기화
classifier = CategoryClassifier(emb_dim=1024, hidden_dim=512).to(device)

# 손실함수: CrossEntropyLoss
#  - forward 결과 x: (batch, 54) => 로짓
#  - label: 0~53 범위 정수
criterion = nn.CrossEntropyLoss()

# 옵티마이저
optimizer = optim.AdamW(classifier.parameters(), lr=1e-4)

In [ ]:
# 기록 주의: 별도 검증셋 없이 학습 배치에서 계산한 지표입니다.
from sklearn.metrics import accuracy_score, f1_score, classification_report

EPOCHS = 120
classifier.train()

for epoch in range(EPOCHS):
    total_loss = 0.0

    # [추가] 각 배치마다 예측값, 정답 레이블 저장
    all_preds = []
    all_labels = []

    for q_emb, label_idx in train_loader:
        # q_emb: (batch, 1024)
        # label_idx: (batch,)   (0~53)

        # 디바이스 이동
        q_emb = q_emb.to(device)
        label_idx = label_idx.to(device)

        optimizer.zero_grad()

        # forward
        logits = classifier(q_emb, category_embeddings)  # (batch, 54)

        loss = criterion(logits, label_idx)

        # backward
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # [추가] 예측값(배치 단위) 계산 -> argmax
        preds = torch.argmax(logits, dim=1)  # (batch,)

        # CPU로 옮기고, numpy로 변환하여 리스트에 추가
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(label_idx.cpu().numpy())

    # 한 epoch가 끝난 시점에서 평균 loss, 정확도, F1, 그리고 클래스별 레포트 계산
    avg_loss = total_loss / len(train_loader)
    epoch_accuracy = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average='macro')

    # [추가] classification_report로 클래스별 세부 지표 출력
    class_report = classification_report(all_labels, all_preds, digits=4)

    print(f"[Epoch {epoch+1}] avg_loss: {avg_loss:.4f} | acc: {epoch_accuracy:.4f} | f1: {epoch_f1:.4f}")

print(class_report)
print("학습 완료!")

[Epoch 1] avg_loss: 3.8576 | acc: 0.1180 | f1: 0.0047


[Epoch 2] avg_loss: 3.6953 | acc: 0.1250 | f1: 0.0041


[Epoch 3] avg_loss: 3.6248 | acc: 0.1250 | f1: 0.0041


[Epoch 4] avg_loss: 3.5354 | acc: 0.1250 | f1: 0.0041


[Epoch 5] avg_loss: 3.4213 | acc: 0.1250 | f1: 0.0041


...

[Epoch 30] avg_loss: 1.1923 | acc: 0.7416 | f1: 0.6762


[Epoch 31] avg_loss: 1.1471 | acc: 0.7547 | f1: 0.7013


[Epoch 32] avg_loss: 1.0984 | acc: 0.7651 | f1: 0.7134


[Epoch 33] avg_loss: 1.0554 | acc: 0.7701 | f1: 0.7228


[Epoch 34] avg_loss: 1.0170 | acc: 0.7882 | f1: 0.7510
[Epoch 35] avg_loss: 0.9785 | acc: 0.7932 | f1: 0.7610
[Epoch 36] avg_loss: 0.9382 | acc: 0.8058 | f1: 0.7777
[Epoch 37] avg_loss: 0.9039 | acc: 0.8135 | f1: 0.7848
[Epoch 38] avg_loss: 0.8740 | acc: 0.8243 | f1: 0.8050
[Epoch 39] avg_loss: 0.8403 | acc: 0.8351 | f1: 0.8210
[Epoch 40] avg_loss: 0.8082 | acc: 0.8351 | f1: 0.8148
[Epoch 41] avg_loss: 0.7805 | acc: 0.8435 | f1: 0.8316
[Epoch 42] avg_loss: 0.7479 | acc: 0.8489 | f1: 0.8386
[Epoch 43] avg_loss: 0.7235 | acc: 0.8581 | f1: 0.8508
[Epoch 44] avg_loss: 0.7015 | acc: 0.8612 | f1: 0.8547
[Epoch 45] avg_loss: 0.6740 | acc: 0.8658 | f1: 0.8581
[Epoch 46] avg_loss: 0.6488 | acc: 0.8731 | f1: 0.8707
[Epoch 47] avg_loss: 0.6273 | acc: 0.8724 | f1: 0.8693
[Epoch 48] avg_loss: 0.6075 | acc: 0.8820 | f1: 0.8779
[Epoch 49] avg_loss: 0.5858 | acc: 0.8858 | f1: 0.8831
[Epoch 50] avg_loss: 0.5686 | acc: 0.8866 | f1: 0.8842
[Epoch 51] avg_loss: 0.5499 | acc: 0.8916 | f1: 0.8921
[Epoch 52]

#### 2-1. 모델 저장

In [ ]:
# 학습 완료 후 모델 저장
torch.save(classifier.state_dict(), '/content/drive/MyDrive/Project_final/model/category_classifier.pth')

print("모델 저장 완료!")

모델 저장 완료!


In [ ]:
# 카테고리 임베딩 리스트를 NumPy 배열로 변환
category_embeddings_np = category_embeddings.cpu().numpy()

# NumPy 배열을 파일로 저장
np.save('/content/drive/MyDrive/Project_final/model/category_embeddings.npy', category_embeddings_np)

### 3. test 코드

In [ ]:
def predict_category(question: str):
    classifier.eval()
    with torch.no_grad():
        # (1, 1024)
        q_emb = embedding_model.encode(question, convert_to_tensor=True).unsqueeze(0).to(device)

        logits = classifier(q_emb, category_embeddings)  # (1, 54)
        probs = torch.softmax(logits, dim=1)             # 확률 분포 (1, 54)

        pred_idx = torch.argmax(probs, dim=1).item()     # 0~53
        return CATEGORIES[pred_idx], probs.cpu().numpy().flatten()

In [ ]:
test_questions = [
    "스타벅스에 자주가는데 할인이 많이 되는 카드를 알려주세요",
    "교통비 할인이 잘되는 카드가 있나요?",
    "인터넷에서 쇼핑을 자주하는데 할인율이 높은 카드를 알려주세요.",
    "카페랑 음식점에서 할인이 잘되는 카드를 알려주세요.",
    "쇼핑할 때 할인 많이 받을 수 있는 카드 추천해줘.",
    "여행 다닐 때 혜택 좋은 카드가 뭐야?",
    "주유소에서 할인 많이 되는 카드 알려줘.",
    "음식점에서 사용하기 좋은 카드 추천해줄래?",
    "영화 할인 혜택 있는 카드 있어?",
    "항공 마일리지 적립 잘 되는 카드 찾고 있어.",
    "편의점에서 쓸만한 카드 뭐야?",
    "교통비 할인 혜택이 있는 카드 추천 부탁해.",
    "외식할 때 사용하기 좋은 카드는 뭐야?",
    "넷플릭스 구독비 할인되는 카드를 추천해주세요.",
    "멜론 월정액 할인되는 카드가 뭔가요",
    "호텔 뷔페를 자주가는데 좋은 카드를 추천해주세요.",
    "호텔에 자주가는데 좋은 카드를 추천해주세요.",
    "쿠팡 멤버쉽 구독비 할인 잘되는 카드가 뭔가요",
    "네이버플러스 멤버십 구독비 할인 되는 카드를 알려주세요",
    "애완동물을 키우는데 좋은 카드를 추천해주세요.",
    "개를 키우는데 좋은 카드가 있을까요?",
    "LPG를 연료로 하는 차를 타고다닙니다. 좋은 카드를 추천해주세요.",
    "LPG 충전할 때 할인이 많이 되는 카드를 추천해주세요.",
    "다음달에 싱가폴에 가는데 좋은 카드를 골라주세요"
]

for q in test_questions:
    pred_cat, score_arr = predict_category(q)
    print(f"질문: {q}")
    print(f"=> 예측 카테고리: {pred_cat}")
    print(f"=> 카테고리별 확률(소프트맥스): {score_arr}")
    print("----")

질문: 스타벅스에 자주가는데 할인이 많이 되는 카드를 알려주세요
=> 예측 카테고리: 카페
=> 카테고리별 확률(소프트맥스): [1.81303818e-07 4.16010089e-08 3.20360268e-05 3.80516917e-06
 7.22006973e-07 4.68630651e-06 6.51329492e-06 3.75019589e-08
 4.71061003e-06 4.02809419e-05 1.97303663e-07 5.73696006e-08
 8.50569037e-09 6.67867027e-07 1.04383889e-06 1.15788202e-06
 9.54612005e-07 5.02693240e-07 4.37571689e-05 2.30904789e-06
 1.78732108e-07 2.09599375e-06 1.30904266e-06 1.47595108e-06
 1.95036364e-06 1.78195648e-07 6.31737066e-06 1.19988943e-06
 3.34448123e-05 1.36806948e-08 6.96907705e-08 4.79360530e-03
 2.20125048e-06 4.50089857e-07 3.06533906e-03 2.48482775e-06
 1.70537945e-07 1.10817176e-07 4.66676966e-07 9.91042018e-01
 3.00260012e-07 9.75476695e-08 2.43563193e-07 2.15323962e-05
 5.22336272e-08 6.76346317e-05 8.08718731e-04 3.44276536e-08
 1.94685782e-07 3.83985430e-08 1.08411831e-08 6.20199643e-08
 2.06439199e-06 4.10058590e-07]
----
질문: 교통비 할인이 잘되는 카드가 있나요?
=> 예측 카테고리: 대중교통
=> 카테고리별 확률(소프트맥스): [1.76027982e-06 3.56347391e-06 9.4254

In [ ]:
def predict_category2(question: str):
    classifier.eval()
    with torch.no_grad():
        # (1, 1024)
        q_emb = embedding_model.encode(question, convert_to_tensor=True).unsqueeze(0).to(device)

        logits = classifier(q_emb, category_embeddings)  # (1, 54)
        probs = torch.softmax(logits, dim=1)             # 확률 분포 (1, 54)

        # 상위 2개 인덱스와 해당 확률값 가져오기
        top2_probs, top2_indices = torch.topk(probs, 2, dim=1)

        # 상위 2개 확률값
        prob1 = top2_probs[0][0].item()
        prob2 = top2_probs[0][1].item()

        # 참고: 평균은 항상 1/54이며 신뢰도 지표가 아닙니다.
        avg_prob = torch.mean(probs).item()

        # 예측 카테고리 가져오기
        pred_cat1 = CATEGORIES[top2_indices[0][0].item()]
        pred_cat2 = CATEGORIES[top2_indices[0][1].item()]

        return [pred_cat1, pred_cat2], [prob1, prob2, avg_prob]

In [ ]:
test_questions = [
    "스타벅스에 자주가는데 할인이 많이 되는 카드를 알려주세요",
    "교통비 할인이 잘되는 카드가 있나요?",
    "인터넷에서 쇼핑을 자주하는데 할인율이 높은 카드를 알려주세요.",
    "카페랑 음식점에서 할인이 잘되는 카드를 알려주세요.",
    "쇼핑할 때 할인 많이 받을 수 있는 카드 추천해줘.",
    "여행 다닐 때 혜택 좋은 카드가 뭐야?",
    "주유소에서 할인 많이 되는 카드 알려줘.",
    "음식점에서 사용하기 좋은 카드 추천해줄래?",
    "영화 할인 혜택 있는 카드 있어?",
    "항공 마일리지 적립 잘 되는 카드 찾고 있어.",
    "편의점에서 쓸만한 카드 뭐야?",
    "교통비 할인 혜택이 있는 카드 추천 부탁해.",
    "외식할 때 사용하기 좋은 카드는 뭐야?",
    "넷플릭스 구독비 할인되는 카드를 추천해주세요.",
    "멜론 월정액 할인되는 카드가 뭔가요",
    "호텔 뷔페를 자주가는데 좋은 카드를 추천해주세요.",
    "호텔에 자주가는데 좋은 카드를 추천해주세요.",
    "쿠팡 멤버쉽 구독비 할인 잘되는 카드가 뭔가요",
    "네이버플러스 멤버십 구독비 할인 되는 카드를 알려주세요",
    "애완동물을 키우는데 좋은 카드를 추천해주세요.",
    "개를 키우는데 좋은 카드가 있을까요?",
    "LPG를 연료로 하는 차를 타고다닙니다. 좋은 카드를 추천해주세요.",
    "LPG 충전할 때 할인이 많이 되는 카드를 추천해주세요.",
    "다음달에 싱가폴에 가는데 좋은 카드를 골라주세요"
]

for q in test_questions:
    pred_cat, score_arr = predict_category2(q)
    print(f"질문: {q}")
    print(f"=> 예측 카테고리: {pred_cat}")
    print(f"=> 카테고리별 확률(소프트맥스): {score_arr}")
    print("----")

질문: 스타벅스에 자주가는데 할인이 많이 되는 카드를 알려주세요
=> 예측 카테고리: ['카페', '음식점']
=> 카테고리별 확률(소프트맥스): [0.9910420179367065, 0.004793605301529169, 0.018518520519137383]
----
질문: 교통비 할인이 잘되는 카드가 있나요?
=> 예측 카테고리: ['대중교통', '해외대중교통']
=> 카테고리별 확률(소프트맥스): [0.9345167279243469, 0.047802213579416275, 0.018518520519137383]
----
질문: 인터넷에서 쇼핑을 자주하는데 할인율이 높은 카드를 알려주세요.
=> 예측 카테고리: ['온라인쇼핑', '홈쇼핑']
=> 카테고리별 확률(소프트맥스): [0.9910304546356201, 0.004455517511814833, 0.018518518656492233]
----
질문: 카페랑 음식점에서 할인이 잘되는 카드를 알려주세요.
=> 예측 카테고리: ['음식점', '카페']
=> 카테고리별 확률(소프트맥스): [0.9423307180404663, 0.040664467960596085, 0.018518520519137383]
----
질문: 쇼핑할 때 할인 많이 받을 수 있는 카드 추천해줘.
=> 예측 카테고리: ['쇼핑', '온라인쇼핑']
=> 카테고리별 확률(소프트맥스): [0.3834731876850128, 0.20157572627067566, 0.018518520519137383]
----
질문: 여행 다닐 때 혜택 좋은 카드가 뭐야?
=> 예측 카테고리: ['여행', '항공사']
=> 카테고리별 확률(소프트맥스): [0.8531474471092224, 0.10109192878007889, 0.018518518656492233]
----
질문: 주유소에서 할인 많이 되는 카드 알려줘.
=> 예측 카테고리: ['주유소', '주차장']
=> 카테고리별 확률(소프트맥스): [0.9138071537017822, 0.0152613

### 4. 모델 불러오기

In [ ]:
import torch
import torch.nn as nn
from sentence_transformers import SentenceTransformer
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [ ]:
# CategoryClassifier 클래스 정의
class CategoryClassifier(nn.Module):
    def __init__(self, emb_dim=1024, hidden_dim=512):
        super().__init__()
        self.fc1 = nn.Linear(emb_dim * 2, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, 1)

    def forward(self, q_emb_batch, cat_emb_all):
        batch_size = q_emb_batch.size(0)
        num_categories = cat_emb_all.size(0)
        q_emb_batch = q_emb_batch.unsqueeze(1)
        q_emb_batch = q_emb_batch.expand(batch_size, num_categories, q_emb_batch.size(-1))
        cat_emb_all = cat_emb_all.unsqueeze(0)
        cat_emb_all = cat_emb_all.expand(batch_size, num_categories, cat_emb_all.size(-1))
        x = torch.cat([q_emb_batch, cat_emb_all], dim=-1)
        x = x.view(batch_size * num_categories, -1)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = x.view(batch_size, num_categories)
        return x

In [ ]:
# CategoryClassifier랑 카테고리 임베딩 벡터 로드
# model_path : CategoryClassifier와 카테고리 임베딩 벡터의 가중치가 있는 폴더의 경로
# device : torch gpu device
def load_classifer(model_path, device):
    classifier = CategoryClassifier(emb_dim=1024, hidden_dim=512)
    classifier.load_state_dict(torch.load(model_path + '/category_classifier.pth'))
    classifier.to(device)

    # NumPy 배열을 파일에서 불러오기
    category_embeddings_np = np.load(model_path + '/category_embeddings.npy')

    # NumPy 배열을 PyTorch 텐서로 변환
    category_embeddings = torch.from_numpy(category_embeddings_np).to(device)

    return classifier, category_embeddings

In [ ]:
# 카테고리 분류 하는 함수
# classifier : 위에서 로드한 CategoryClassifier
# category_embeddings : 위에서 로드한 카테고리 임베딩 벡터
# embedding_model : 미리 로드해둔 "intfloat/multilingual-e5-large-instruct"
# device : torch gpu device
def classify_category(classifier, category_embeddings, embedding_model, device, question: str):
    # 54개 카테고리
    CATEGORIES = ['LPG충전소', 'OTT', '간편결제', '게임', '공과금', '공연', '기업형슈퍼마켓', '대중교통',
          '대형마트', '드럭스토어', '디지털구독', '렌탈', '리조트', '멤버십구독', '면세점', '미용', '미용실',
          '반려동물', '배달', '백화점', '보험', '세탁소', '쇼핑', '스포츠', '아울렛', '여행', '영화관',
          '온라인서점', '온라인쇼핑', '우체국', '웹툰', '음식점', '의료', '전기차충전소', '제과아이스크림',
          '주유소', '주차장', '차량정비소', '창고형할인매장', '카페', '콘도', '타이어샵', '택시', '테마파크',
          '통신사', '패션', '편의점', '학습지', '학원', '항공사', '해외대중교통', '호텔', '호텔음식점',
          '홈쇼핑']

    classifier.eval()
    with torch.no_grad():
        # (1, 1024)
        q_emb = embedding_model.encode(question, convert_to_tensor=True).unsqueeze(0).to(device)

        logits = classifier(q_emb, category_embeddings)  # (1, 54)
        probs = torch.softmax(logits, dim=1)             # 확률 분포 (1, 54)

        pred_idx = torch.argmax(probs, dim=1).item()     # 0~53
        return CATEGORIES[pred_idx]

In [ ]:
# 임베딩 모델 load
embedding_model = SentenceTransformer("intfloat/multilingual-e5-large-instruct").to(device)

# CategoryClassifier랑 카테고리 임베딩 벡터 로드
classifier, category_embeddings = load_classifer('/content/drive/MyDrive/Project_final/model', device)

In [ ]:
test_questions = [
    "스타벅스에 자주가는데 할인이 많이 되는 카드를 알려주세요",
    "교통비 할인이 잘되는 카드가 있나요?",
    "인터넷에서 쇼핑을 자주하는데 할인율이 높은 카드를 알려주세요.",
    "카페랑 음식점에서 할인이 잘되는 카드를 알려주세요.",
    "쇼핑할 때 할인 많이 받을 수 있는 카드 추천해줘.",
    "여행 다닐 때 혜택 좋은 카드가 뭐야?",
    "주유소에서 할인 많이 되는 카드 알려줘.",
    "음식점에서 사용하기 좋은 카드 추천해줄래?",
    "영화 할인 혜택 있는 카드 있어?",
    "항공 마일리지 적립 잘 되는 카드 찾고 있어.",
    "편의점에서 쓸만한 카드 뭐야?",
    "교통비 할인 혜택이 있는 카드 추천 부탁해.",
    "외식할 때 사용하기 좋은 카드는 뭐야?",
    "넷플릭스 구독비 할인되는 카드를 추천해주세요.",
    "멜론 월정액 할인되는 카드가 뭔가요",
    "호텔 뷔페를 자주가는데 좋은 카드를 추천해주세요.",
    "호텔에 자주가는데 좋은 카드를 추천해주세요.",
    "쿠팡 멤버쉽 구독비 할인 잘되는 카드가 뭔가요",
    "네이버플러스 멤버십 구독비 할인 되는 카드를 알려주세요",
    "애완동물을 키우는데 좋은 카드를 추천해주세요.",
    "개를 키우는데 좋은 카드가 있을까요?",
    "LPG를 연료로 하는 차를 타고다닙니다. 좋은 카드를 추천해주세요.",
    "LPG 충전할 때 할인이 많이 되는 카드를 추천해주세요.",
    "다음달에 싱가폴에 가는데 좋은 카드를 골라주세요",
    "밖에서 밥을 자주 사먹는데 사용하기 좋은 카드를 골라줘",
    "쿠팡와우 구독비 줄이고싶은데 좋은 카드가 있을까요"
]

for q in test_questions:
    pred_cat = classify_category(classifier, category_embeddings, embedding_model, device, q)
    print(f"질문: {q}")
    print(f"=> 예측 카테고리: {pred_cat}")
    print("----")

질문: 스타벅스에 자주가는데 할인이 많이 되는 카드를 알려주세요
=> 예측 카테고리: 카페
----
질문: 교통비 할인이 잘되는 카드가 있나요?
=> 예측 카테고리: 대중교통
----
질문: 인터넷에서 쇼핑을 자주하는데 할인율이 높은 카드를 알려주세요.
=> 예측 카테고리: 온라인쇼핑
----
질문: 카페랑 음식점에서 할인이 잘되는 카드를 알려주세요.
=> 예측 카테고리: 음식점
----
질문: 쇼핑할 때 할인 많이 받을 수 있는 카드 추천해줘.
=> 예측 카테고리: 쇼핑
----
질문: 여행 다닐 때 혜택 좋은 카드가 뭐야?
=> 예측 카테고리: 여행
----
질문: 주유소에서 할인 많이 되는 카드 알려줘.
=> 예측 카테고리: 주유소
----
질문: 음식점에서 사용하기 좋은 카드 추천해줄래?
=> 예측 카테고리: 음식점
----
질문: 영화 할인 혜택 있는 카드 있어?
=> 예측 카테고리: 영화관
----
질문: 항공 마일리지 적립 잘 되는 카드 찾고 있어.
=> 예측 카테고리: 항공사
----
질문: 편의점에서 쓸만한 카드 뭐야?
=> 예측 카테고리: 편의점
----
질문: 교통비 할인 혜택이 있는 카드 추천 부탁해.
=> 예측 카테고리: 대중교통
----
질문: 외식할 때 사용하기 좋은 카드는 뭐야?
=> 예측 카테고리: 음식점
----
질문: 넷플릭스 구독비 할인되는 카드를 추천해주세요.
=> 예측 카테고리: OTT
----
질문: 멜론 월정액 할인되는 카드가 뭔가요
=> 예측 카테고리: 디지털구독
----
질문: 호텔 뷔페를 자주가는데 좋은 카드를 추천해주세요.
=> 예측 카테고리: 호텔음식점
----
질문: 호텔에 자주가는데 좋은 카드를 추천해주세요.
=> 예측 카테고리: 호텔
----
질문: 쿠팡 멤버쉽 구독비 할인 잘되는 카드가 뭔가요
=> 예측 카테고리: 멤버십구독
----
질문: 네이버플러스 멤버십 구독비 할인 되는 카드를 알려주세요
=> 예측 카테고리: 멤버십구독
----
질문: 애완동물을 키우는데 좋은 카드를 추천해주세요.
=> 예측 카